
# DV Photonic Noise Study for MerLin/Perceval QPINN (Phase 1 + Phase 2)

This notebook studies physically correct **discrete-variable (DV) photonic noise** for the MerLin/Perceval quantum layers used in `daniel_phase_1.ipynb` and `daniel_phase_2.ipynb`.

Primary focus is DV/Fock-space effects:
- finite-shot sampling noise,
- photon loss (transmittance),
- detector model (PNR vs threshold),
- phase-setting noise,
- source imperfections (brightness, indistinguishability, `g2`).

CV-specific noise (homodyne/quadrature/squeezing/displacement/Kerr-style CV gates) is **not** the primary model for this implementation.


In [ ]:

import os
import time
import copy
import math
import random
import warnings
from dataclasses import dataclass
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy.interpolate import RegularGridInterpolator

import merlin as ML
import perceval as pcvl

print('Torch:', torch.__version__)
print('MerLin:', getattr(ML, '__version__', 'unknown'))
print('Perceval:', getattr(pcvl, '__version__', 'unknown'))

DEVICE = torch.device('cpu')
DTYPE = torch.float32
torch.set_default_dtype(DTYPE)

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

FAST_MODE = True

CONFIG = {
    'phase1_train': dict(epochs=80 if FAST_MODE else 300, n_f=64, n_i=64, n_b=64, lr=1e-2, lambda_pde=1.0, lambda_ic=10.0, lambda_bc=1.0, lambda_consistency=0.1),
    'phase2_train': dict(epochs=60 if FAST_MODE else 250, n_f=96, n_i=96, n_b=96, lr=1e-2, lambda_pde=1.0, lambda_ic=10.0, lambda_bc=1.0, lambda_consistency=0.1),
    'eval_grid': dict(nx=61 if FAST_MODE else 101, nt=61 if FAST_MODE else 101),
    'shots_values': [100, 250, 500, 1000, 2500, 5000, 10000] if not FAST_MODE else [100, 500, 1000, 2500],
    'shot_seeds': [0, 1, 2] if FAST_MODE else [0, 1, 2, 3, 4],
    'transmittance_values': [1.0, 0.98, 0.95, 0.90, 0.85, 0.80, 0.70],
    'phase_values': [0.0, 0.005, 0.01, 0.02, 0.05, 0.10],
    'brightness_values': [1.0, 0.98, 0.95, 0.90, 0.85],
    'indist_values': [1.0, 0.98, 0.95, 0.90, 0.80],
    'g2_values': [0.0, 0.001, 0.005, 0.01, 0.02],
}

os.makedirs('results', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)
print('FAST_MODE =', FAST_MODE)


## Reused Phase 1 and Phase 2 Model/Data Definitions

In [ ]:

# -------------------------
# Phase 1 (heat equation)
# -------------------------
alpha1 = 0.1

def exact_u1(x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    return torch.exp(-alpha1 * math.pi**2 * t) * torch.sin(math.pi * x)

def sample_interior1(n: int) -> torch.Tensor:
    x = torch.rand(n, 1, device=DEVICE, dtype=DTYPE)
    t = torch.rand(n, 1, device=DEVICE, dtype=DTYPE)
    xt = torch.cat([x, t], dim=1)
    xt.requires_grad_(True)
    return xt

def sample_initial1(n: int):
    x = torch.rand(n, 1, device=DEVICE, dtype=DTYPE)
    t = torch.zeros_like(x)
    xt = torch.cat([x, t], dim=1)
    y = exact_u1(x, t)
    return xt, y

def sample_boundary1(n: int):
    n0 = n // 2
    n1 = n - n0
    t0 = torch.rand(n0, 1, device=DEVICE, dtype=DTYPE)
    t1 = torch.rand(n1, 1, device=DEVICE, dtype=DTYPE)
    x0 = torch.zeros_like(t0)
    x1 = torch.ones_like(t1)
    xt = torch.cat([torch.cat([x0, t0], dim=1), torch.cat([x1, t1], dim=1)], dim=0)
    y = torch.zeros(n, 1, device=DEVICE, dtype=DTYPE)
    return xt, y

def gradients(y: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), create_graph=True, retain_graph=True)[0]

class MerlinQPINNPhase1(nn.Module):
    def __init__(self, feature_size: int = 4, quantum_output_size: int = 4, hidden: int = 16):
        super().__init__()
        self.feature_size = feature_size
        self.quantum_output_size = quantum_output_size
        self.feature_map = nn.Sequential(nn.Linear(2, hidden), nn.Tanh(), nn.Linear(hidden, feature_size))
        self.quantum = ML.QuantumLayer.simple(input_size=feature_size, output_size=quantum_output_size)
        self.readout = nn.Sequential(nn.Linear(quantum_output_size, hidden), nn.Tanh(), nn.Linear(hidden, 2))
        self.quantum_build_info = {
            'origin': 'QuantumLayer.simple',
            'uses_circuit_builder': False,
            'uses_pcvl_circuit': True,
            'uses_pcvl_experiment': True,
        }

    def forward(self, xt: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = xt[:, 0:1]
        z = self.feature_map(xt)
        q = self.quantum(z)
        out = self.readout(q)
        q_u = out[:, 0:1]
        ux_hat = out[:, 1:2]
        u = x * (1.0 - x) * q_u
        return u, ux_hat


def pde_residual_phase1(model, xt):
    u, ux_hat = model(xt)
    grad_u = gradients(u, xt)
    u_x = grad_u[:, 0:1]
    u_t = grad_u[:, 1:2]
    ux_hat_x = gradients(ux_hat, xt)[:, 0:1]
    residual = u_t - alpha1 * ux_hat_x
    consistency = u_x - ux_hat
    return residual, consistency

# -------------------------
# Phase 2 (dataset and model reused style)
# -------------------------
ALPHA2 = 0.30
XMIN2, XMAX2 = -math.pi / 2, math.pi / 2
TMIN2, TMAX2 = 0.0, 0.5
SIGMA2 = 0.2
X0 = -math.pi / 8

def initial_condition2(x: np.ndarray) -> np.ndarray:
    return 0.5 * np.exp(-(x - X0) ** 2 / (2 * SIGMA2))

def solve_reference2(nx=301, nt_eval=301):
    x = np.linspace(XMIN2, XMAX2, nx)
    dx = x[1] - x[0]
    u0 = initial_condition2(x)
    u0_interior = u0[1:-1].astype(np.float64)
    n = nx - 2
    main = -2.0 * np.ones(n)
    off = 1.0 * np.ones(n - 1)
    L = (np.diag(main) + np.diag(off, 1) + np.diag(off, -1)) / (dx ** 2)
    def rhs(t, u):
        return ALPHA2 * (L @ u)
    t_eval = np.linspace(TMIN2, TMAX2, nt_eval)
    sol = solve_ivp(rhs, (TMIN2, TMAX2), u0_interior, t_eval=t_eval, method='RK45')
    U = np.zeros((nx, nt_eval))
    U[1:-1, :] = sol.y
    U[0, :] = 0.0
    U[-1, :] = 0.0
    return x, t_eval, U

x_ref2, t_ref2, U_ref2 = solve_reference2(nx=301 if FAST_MODE else 401, nt_eval=301 if FAST_MODE else 401)
interp_ref2 = RegularGridInterpolator((x_ref2, t_ref2), U_ref2, bounds_error=False, fill_value=0.0)

def reference_at2(x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
    pts = np.stack([x.detach().cpu().numpy().ravel(), t.detach().cpu().numpy().ravel()], axis=1)
    vals = interp_ref2(pts)
    return torch.tensor(vals, dtype=DTYPE, device=DEVICE).view(-1, 1)

def bc_env2(x: torch.Tensor) -> torch.Tensor:
    a = math.pi / 2
    return a * a - x * x

def bc_env2_d1(x: torch.Tensor) -> torch.Tensor:
    return -2.0 * x

class MerlinAuxQPINNPhase2(nn.Module):
    def __init__(self, feature_size=4, quantum_output_size=4, hidden=16, depth=1):
        super().__init__()
        self.feature_size = feature_size
        self.quantum_output_size = quantum_output_size
        self.depth = depth
        self.feature_map = nn.Sequential(nn.Linear(2, hidden), nn.Tanh(), nn.Linear(hidden, feature_size))

        if depth == 1:
            self.quantum = ML.QuantumLayer.simple(input_size=feature_size, output_size=quantum_output_size)
            self.post = nn.Identity()
            build_info = dict(origin='QuantumLayer.simple', uses_circuit_builder=False, uses_pcvl_circuit=True, uses_pcvl_experiment=True)
        else:
            n_modes = feature_size + 1
            n_photons = math.ceil((feature_size + 1) / 2)
            builder = ML.CircuitBuilder(n_modes=n_modes)
            for i in range(depth):
                builder.add_entangling_layer(trainable=True, name=f'L{i}')
            builder.add_angle_encoding(modes=list(range(feature_size)), name='input')
            for i in range(depth):
                builder.add_entangling_layer(trainable=True, name=f'R{i}')
            circ = builder.to_pcvl_circuit()
            exp = pcvl.Experiment(m_circuit=circ)
            state = [0] * n_modes
            for k in range(n_photons):
                state[2 * k] = 1
            exp.with_input(pcvl.BasicState(state))

            self.quantum = ML.QuantumLayer(
                experiment=exp,
                n_photons=n_photons,
                trainable_parameters=[f'L{i}' for i in range(depth)] + [f'R{i}' for i in range(depth)],
                input_parameters=['input'],
                measurement_strategy=ML.MeasurementStrategy.probs(),
            )
            q_out = self.quantum.output_size
            self.post = ML.ModGrouping(input_size=q_out, output_size=quantum_output_size) if q_out != quantum_output_size else nn.Identity()
            build_info = dict(origin='CircuitBuilder -> pcvl.Circuit -> pcvl.Experiment -> QuantumLayer', uses_circuit_builder=True, uses_pcvl_circuit=True, uses_pcvl_experiment=True)

        self.quantum_build_info = build_info
        self.readout = nn.Sequential(nn.Linear(quantum_output_size, hidden), nn.Tanh(), nn.Linear(hidden, 2))

    def forward(self, xt):
        x = xt[:, 0:1]
        z = self.feature_map(xt)
        q = self.quantum(z)
        if self.depth > 1:
            q = self.post(q)
        out = self.readout(q)
        q_u = out[:, 0:1]
        qx_hat = out[:, 1:2]
        env = bc_env2(x)
        env_d1 = bc_env2_d1(x)
        T = env * q_u
        Tx = env_d1 * q_u + env * qx_hat
        return T, Tx


def sample_interior2(n):
    x = torch.empty(n, 1, device=DEVICE, dtype=DTYPE).uniform_(XMIN2, XMAX2)
    t = torch.empty(n, 1, device=DEVICE, dtype=DTYPE).uniform_(TMIN2, TMAX2)
    xt = torch.cat([x, t], dim=1)
    xt.requires_grad_(True)
    return xt

def sample_initial2(n):
    x = torch.empty(n, 1, device=DEVICE, dtype=DTYPE).uniform_(XMIN2, XMAX2)
    t = torch.full((n, 1), TMIN2, device=DEVICE, dtype=DTYPE)
    xt = torch.cat([x, t], dim=1)
    y = reference_at2(x, torch.zeros_like(x))
    return xt, y

def sample_boundary2(n):
    side = torch.rand(n, 1, device=DEVICE, dtype=DTYPE)
    x = torch.where(side < 0.5, torch.full_like(side, XMIN2), torch.full_like(side, XMAX2))
    t = torch.empty(n, 1, device=DEVICE, dtype=DTYPE).uniform_(TMIN2, TMAX2)
    xt = torch.cat([x, t], dim=1)
    y = torch.zeros(n, 1, device=DEVICE, dtype=DTYPE)
    return xt, y


def pde_residual_phase2(model, xt):
    xt = xt.requires_grad_(True)
    T, Tx = model(xt)
    T_t = gradients(T, xt)[:, 1:2]
    Tx_x = gradients(Tx, xt)[:, 0:1]
    residual = T_t - ALPHA2 * Tx_x
    T_x_auto = gradients(T, xt)[:, 0:1]
    consistency = T_x_auto - Tx
    return residual, consistency

print('Loaded model definitions and reference data.')


In [ ]:

@dataclass
class TrainConfig:
    epochs: int
    n_f: int
    n_i: int
    n_b: int
    lr: float
    lambda_pde: float
    lambda_ic: float
    lambda_bc: float
    lambda_consistency: float


def train_phase_model(model, phase: str, cfg: TrainConfig):
    model = model.to(device=DEVICE, dtype=DTYPE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    mse = nn.MSELoss()
    history = {'total': [], 'pde': [], 'ic': [], 'bc': [], 'consistency': []}
    t0 = time.time()

    for epoch in range(1, cfg.epochs + 1):
        opt.zero_grad()
        if phase == 'phase1':
            xt_f = sample_interior1(cfg.n_f)
            xt_i, y_i = sample_initial1(cfg.n_i)
            xt_b, y_b = sample_boundary1(cfg.n_b)
            r_f, r_c = pde_residual_phase1(model, xt_f)
            u_i, _ = model(xt_i)
            u_b, _ = model(xt_b)
        else:
            xt_f = sample_interior2(cfg.n_f)
            xt_i, y_i = sample_initial2(cfg.n_i)
            xt_b, y_b = sample_boundary2(cfg.n_b)
            r_f, r_c = pde_residual_phase2(model, xt_f)
            u_i, _ = model(xt_i)
            u_b, _ = model(xt_b)

        loss_pde = mse(r_f, torch.zeros_like(r_f))
        loss_ic = mse(u_i, y_i)
        loss_bc = mse(u_b, y_b)
        loss_cons = mse(r_c, torch.zeros_like(r_c))
        loss_total = cfg.lambda_pde * loss_pde + cfg.lambda_ic * loss_ic + cfg.lambda_bc * loss_bc + cfg.lambda_consistency * loss_cons
        loss_total.backward()
        opt.step()

        history['total'].append(loss_total.item())
        history['pde'].append(loss_pde.item())
        history['ic'].append(loss_ic.item())
        history['bc'].append(loss_bc.item())
        history['consistency'].append(loss_cons.item())

    return history, time.time() - t0


def evaluate_phase_metrics(model, phase: str, cfg: TrainConfig, nx=61, nt=61) -> Dict[str, float]:
    t0 = time.time()
    model.eval()

    with torch.no_grad():
        if phase == 'phase1':
            x = torch.linspace(0, 1, nx, device=DEVICE, dtype=DTYPE).view(-1, 1)
            t = torch.linspace(0, 1, nt, device=DEVICE, dtype=DTYPE).view(-1, 1)
            X, T = torch.meshgrid(x.squeeze(), t.squeeze(), indexing='ij')
            xt = torch.stack([X.reshape(-1), T.reshape(-1)], dim=1)
            u_pred, _ = model(xt)
            u_true = exact_u1(X, T)
            pred_grid = u_pred.reshape(nx, nt)
            true_grid = u_true
        else:
            x = torch.linspace(XMIN2, XMAX2, nx, device=DEVICE, dtype=DTYPE).view(-1, 1)
            t = torch.linspace(TMIN2, TMAX2, nt, device=DEVICE, dtype=DTYPE).view(-1, 1)
            X, T = torch.meshgrid(x.squeeze(), t.squeeze(), indexing='ij')
            xt = torch.stack([X.reshape(-1), T.reshape(-1)], dim=1)
            pred, _ = model(xt)
            pred_grid = pred.reshape(nx, nt)
            pts = np.stack([X.detach().cpu().numpy().ravel(), T.detach().cpu().numpy().ravel()], axis=1)
            true_grid = torch.tensor(interp_ref2(pts).reshape(nx, nt), dtype=DTYPE, device=DEVICE)

        diff = pred_grid - true_grid
        rmse = torch.sqrt(torch.mean(diff ** 2)).item()
        mae = torch.mean(torch.abs(diff)).item()

    if phase == 'phase1':
        xt_f = sample_interior1(min(2000, nx * nt))
        xt_i, y_i = sample_initial1(500)
        xt_b, y_b = sample_boundary1(500)
        r_f, r_c = pde_residual_phase1(model, xt_f)
        u_i, _ = model(xt_i)
        u_b, _ = model(xt_b)
    else:
        xt_f = sample_interior2(min(2000, nx * nt))
        xt_i, y_i = sample_initial2(500)
        xt_b, y_b = sample_boundary2(500)
        r_f, r_c = pde_residual_phase2(model, xt_f)
        u_i, _ = model(xt_i)
        u_b, _ = model(xt_b)

    mse = nn.MSELoss()
    loss_pde = mse(r_f, torch.zeros_like(r_f)).item()
    loss_ic = mse(u_i, y_i).item()
    loss_bc = mse(u_b, y_b).item()
    loss_cons = mse(r_c, torch.zeros_like(r_c)).item()
    loss_total = cfg.lambda_pde * loss_pde + cfg.lambda_ic * loss_ic + cfg.lambda_bc * loss_bc + cfg.lambda_consistency * loss_cons

    return {
        'rmse': rmse,
        'mae': mae,
        'pde_residual_loss': loss_pde,
        'ic_loss': loss_ic,
        'bc_loss': loss_bc,
        'consistency_loss': loss_cons,
        'total_loss': loss_total,
        'runtime_s': time.time() - t0,
    }


## Load or Train Phase 1 and Phase 2 Quantum Models

In [ ]:

phase1_ckpt = 'results/phase1_merlin_for_noise.pt'
phase2_ckpt = 'results/phase2_merlin_for_noise.pt'

cfg1 = TrainConfig(**CONFIG['phase1_train'])
cfg2 = TrainConfig(**CONFIG['phase2_train'])

model_phase1 = MerlinQPINNPhase1(feature_size=4, quantum_output_size=4, hidden=16)
model_phase2 = MerlinAuxQPINNPhase2(feature_size=4, quantum_output_size=4, hidden=16, depth=1)

if os.path.exists(phase1_ckpt):
    model_phase1.load_state_dict(torch.load(phase1_ckpt, map_location='cpu'))
    print('Loaded trained Phase 1 weights from', phase1_ckpt)
else:
    warnings.warn('Phase 1 weights not found. Training lightweight model for this notebook.')
    hist1, t1 = train_phase_model(model_phase1, 'phase1', cfg1)
    torch.save(model_phase1.state_dict(), phase1_ckpt)
    print(f'Trained Phase 1 model in {t1:.2f}s and saved checkpoint.')

if os.path.exists(phase2_ckpt):
    model_phase2.load_state_dict(torch.load(phase2_ckpt, map_location='cpu'))
    print('Loaded trained Phase 2 weights from', phase2_ckpt)
else:
    warnings.warn('Phase 2 weights not found. Training lightweight model for this notebook.')
    hist2, t2 = train_phase_model(model_phase2, 'phase2', cfg2)
    torch.save(model_phase2.state_dict(), phase2_ckpt)
    print(f'Trained Phase 2 model in {t2:.2f}s and saved checkpoint.')


## Baseline Sanity Check (Noiseless)

In [ ]:

def _set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

base1 = evaluate_phase_metrics(model_phase1, 'phase1', cfg1, **CONFIG['eval_grid'])
base2 = evaluate_phase_metrics(model_phase2, 'phase2', cfg2, **CONFIG['eval_grid'])

baseline_df = pd.DataFrame([
    {'phase': 'phase1', **base1},
    {'phase': 'phase2', **base2},
])

baseline_df


## Quantum Layer Inspection Report

In [ ]:

def _unwrap_quantum_module(quantum_module: nn.Module):
    if hasattr(quantum_module, 'quantum_layer'):
        core = quantum_module.quantum_layer
        post = getattr(quantum_module, 'post_processing', nn.Identity())
        target_out = getattr(quantum_module, 'output_size', None)
        if target_out is None:
            target_out = getattr(post, 'output_size', getattr(core, 'output_size', None))
        return core, post, target_out
    return quantum_module, nn.Identity(), getattr(quantum_module, 'output_size', None)


def inspect_quantum_layer(model: nn.Module, name: str, n_test: int = 8) -> Dict[str, Any]:
    qmod = model.quantum
    core, post, target_out = _unwrap_quantum_module(qmod)
    exp = getattr(core, 'experiment', None)

    input_state = getattr(core, 'input_state', None)
    n_photons = getattr(core, 'n_photons', None)
    comp_space = getattr(core, 'computation_space', None)
    ms = getattr(core, 'measurement_strategy', None)
    output_keys = getattr(core, 'output_keys', None)
    trainable = getattr(core, 'trainable_parameters', None)
    input_params = getattr(core, 'input_parameters', None)

    x = torch.randn(n_test, getattr(model, 'feature_size', 4), dtype=DTYPE)
    with torch.no_grad():
        q_raw = core(x)
    sums = q_raw.sum(dim=1).detach().cpu().numpy()

    report = {
        'model': name,
        'n_modes': int(qmod.circuit.m) if hasattr(qmod, 'circuit') else (int(exp.m) if exp is not None else None),
        'input_state': str(input_state) if input_state is not None else 'None/encoded via layer',
        'n_photons': int(n_photons) if n_photons is not None else None,
        'computation_space': str(comp_space) if comp_space is not None else 'unknown',
        'measurement_strategy': str(ms.type) if ms is not None and hasattr(ms, 'type') else str(ms),
        'measurement_detail': str(ms) if ms is not None else 'unknown',
        'build_origin': model.quantum_build_info.get('origin', 'unknown') if hasattr(model, 'quantum_build_info') else 'unknown',
        'from_quantumlayer_simple': bool('simple' in str(model.quantum_build_info.get('origin', '')).lower()) if hasattr(model, 'quantum_build_info') else False,
        'from_circuitbuilder': bool(model.quantum_build_info.get('uses_circuit_builder', False)) if hasattr(model, 'quantum_build_info') else False,
        'uses_pcvl_circuit': bool(model.quantum_build_info.get('uses_pcvl_circuit', True)) if hasattr(model, 'quantum_build_info') else True,
        'uses_pcvl_experiment': bool(model.quantum_build_info.get('uses_pcvl_experiment', True)) if hasattr(model, 'quantum_build_info') else True,
        'custom_noise_active': bool(exp is not None and getattr(exp, 'noise', None) is not None),
        'custom_detectors_active': bool(exp is not None and any(d is not None for d in getattr(exp, 'detectors', []))),
        'output_size_before_post': int(getattr(core, 'output_size', -1)),
        'output_size_after_post': int(target_out) if target_out is not None else None,
        'output_keys_len_before_post': len(output_keys) if output_keys is not None else None,
        'prob_sum_mean': float(np.mean(sums)),
        'prob_sum_std': float(np.std(sums)),
        'prob_sum_min': float(np.min(sums)),
        'prob_sum_max': float(np.max(sums)),
        'trainable_parameters': str(trainable),
        'input_parameters': str(input_params),
    }
    return report

inspection_rows = [
    inspect_quantum_layer(model_phase1, 'Phase1_MerlinQPINN'),
    inspect_quantum_layer(model_phase2, 'Phase2_MerlinAuxQPINN_depth1'),
]
inspection_df = pd.DataFrame(inspection_rows)
inspection_df


## Noise Wrapper Utilities

In [ ]:

class FixedQuantumWrapper(nn.Module):
    def __init__(self, core_layer: nn.Module, post_layer: nn.Module, shots: Optional[int] = None, sampling_method: str = 'multinomial'):
        super().__init__()
        self.core_layer = core_layer
        self.post_layer = post_layer
        self.shots = shots
        self.sampling_method = sampling_method

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.shots is None or self.shots == 0:
            q = self.core_layer(x)
        else:
            q = self.core_layer(x, shots=int(self.shots), sampling_method=self.sampling_method)
        return self.post_layer(q)


def build_experiment_from_layer_or_circuit(layer_or_circuit, input_state=None):
    if isinstance(layer_or_circuit, pcvl.Circuit):
        circ = layer_or_circuit
    elif hasattr(layer_or_circuit, 'circuit'):
        circ = layer_or_circuit.circuit
    else:
        raise ValueError('Cannot extract pcvl.Circuit from provided object.')

    exp = pcvl.Experiment(m_circuit=circ)
    if input_state is not None:
        exp.with_input(input_state)
    return exp


def apply_noise_model(experiment: pcvl.Experiment, **noise_kwargs):
    filtered = {k: v for k, v in noise_kwargs.items() if v is not None}
    if len(filtered) == 0:
        return experiment, {'supported': True, 'applied': None}
    try:
        experiment.noise = pcvl.NoiseModel(**filtered)
        return experiment, {'supported': True, 'applied': filtered}
    except Exception as e:
        return experiment, {'supported': False, 'applied': filtered, 'reason': str(e)}


def verify_distribution_changes(base_quantum_module: nn.Module, noisy_quantum_module: nn.Module, feature_size: int, n_samples: int = 16):
    with torch.no_grad():
        x = torch.randn(n_samples, feature_size, dtype=DTYPE)
        y0 = base_quantum_module(x)
        y1 = noisy_quantum_module(x)
    changed_dim = y0.shape[1] != y1.shape[1]
    if changed_dim:
        min_dim = min(y0.shape[1], y1.shape[1])
        delta = torch.mean(torch.abs(y0[:, :min_dim] - y1[:, :min_dim])).item()
    else:
        delta = torch.mean(torch.abs(y0 - y1)).item()
    return {
        'changed_output_dim': changed_dim,
        'base_dim': int(y0.shape[1]),
        'noisy_dim': int(y1.shape[1]),
        'mean_abs_delta_on_overlap': float(delta),
        'distribution_changed': bool(changed_dim or delta > 1e-7),
    }


def _make_noisy_quantum_from_base(base_quantum_module: nn.Module, feature_size: int, target_output_size: int,
                                  shots: Optional[int] = None,
                                  noise_kwargs: Optional[Dict[str, Any]] = None,
                                  detector_type: Optional[str] = None,
                                  force_fock_for_detector: bool = False):
    base_core, _, _ = _unwrap_quantum_module(base_quantum_module)

    exp = build_experiment_from_layer_or_circuit(base_quantum_module, input_state=getattr(base_core, 'input_state', None))

    noise_kwargs = noise_kwargs or {}
    exp, noise_report = apply_noise_model(exp, **noise_kwargs)
    if not noise_report.get('supported', False):
        return None, {'supported': False, 'reason': f"NoiseModel unsupported: {noise_report.get('reason')}"}

    if detector_type is not None:
        try:
            for m in range(exp.m):
                if detector_type.lower() == 'threshold':
                    exp.add(m, pcvl.Detector.threshold())
                elif detector_type.lower() == 'pnr':
                    exp.add(m, pcvl.Detector.pnr())
                else:
                    raise ValueError(f'Unknown detector_type={detector_type}')
        except Exception as e:
            return None, {'supported': False, 'reason': f'Failed to set detectors: {e}'}

    ms = getattr(base_core, 'measurement_strategy', ML.MeasurementStrategy.probs())
    if force_fock_for_detector:
        ms = ML.MeasurementStrategy.probs(computation_space=ML.ComputationSpace.FOCK)

    try:
        new_core = ML.QuantumLayer(
            experiment=exp,
            n_photons=getattr(base_core, 'n_photons', None),
            trainable_parameters=getattr(base_core, 'trainable_parameters', None),
            input_parameters=getattr(base_core, 'input_parameters', None),
            measurement_strategy=ms,
        )
    except Exception as e:
        return None, {'supported': False, 'reason': f'Failed to build noisy QuantumLayer: {e}'}

    try:
        miss = new_core.load_state_dict(base_core.state_dict(), strict=False)
        load_info = {'missing': miss.missing_keys, 'unexpected': miss.unexpected_keys}
    except Exception as e:
        load_info = {'warning': f'state_dict copy issue: {e}'}

    if int(new_core.output_size) != int(target_output_size):
        post = ML.ModGrouping(input_size=int(new_core.output_size), output_size=int(target_output_size))
    else:
        post = nn.Identity()

    wrapped = FixedQuantumWrapper(new_core, post, shots=shots)
    support = {
        'supported': True,
        'noise_report': noise_report,
        'detector_type': detector_type,
        'computation_space': str(new_core.computation_space),
        'new_core_output_size': int(new_core.output_size),
        'target_output_size': int(target_output_size),
        'output_keys_len': len(getattr(new_core, 'output_keys', [])),
        'load_info': load_info,
    }
    return wrapped, support


def evaluate_quantum_model_under_noise(model: nn.Module, phase: str, cfg: TrainConfig,
                                       shots: Optional[int] = None,
                                       noise_kwargs: Optional[Dict[str, Any]] = None,
                                       detector_type: Optional[str] = None,
                                       force_fock_for_detector: bool = False,
                                       seed: int = 0):
    _set_seed(seed)
    model_noisy = copy.deepcopy(model)

    base_quantum = model_noisy.quantum
    target_q_out = getattr(model_noisy, 'quantum_output_size', 4)
    feature_size = getattr(model_noisy, 'feature_size', 4)

    wrapped, support = _make_noisy_quantum_from_base(
        base_quantum_module=base_quantum,
        feature_size=feature_size,
        target_output_size=target_q_out,
        shots=shots,
        noise_kwargs=noise_kwargs,
        detector_type=detector_type,
        force_fock_for_detector=force_fock_for_detector,
    )

    if wrapped is None:
        return {'supported': False, 'phase': phase, 'metrics': None, 'support': support}

    model_noisy.quantum = wrapped

    dist_change = verify_distribution_changes(base_quantum, wrapped, feature_size=feature_size, n_samples=12)

    try:
        metrics = evaluate_phase_metrics(model_noisy, phase=phase, cfg=cfg, **CONFIG['eval_grid'])
        return {
            'supported': True,
            'phase': phase,
            'metrics': metrics,
            'support': support,
            'distribution_change': dist_change,
        }
    except Exception as e:
        return {
            'supported': False,
            'phase': phase,
            'metrics': None,
            'support': {**support, 'reason': f'evaluation failed: {e}'},
            'distribution_change': dist_change,
        }


def summarize_metrics(rows: List[Dict[str, Any]]) -> pd.DataFrame:
    if len(rows) == 0:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    metric_cols = ['rmse', 'mae', 'pde_residual_loss', 'ic_loss', 'bc_loss', 'consistency_loss', 'total_loss', 'runtime_s']
    for c in metric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    keys = [k for k in ['phase', 'noise_family', 'setting', 'value', 'supported'] if k in df.columns]
    agg = df.groupby(keys, dropna=False)[metric_cols].agg(['mean', 'std']).reset_index()
    agg.columns = ['_'.join(col).strip('_') for col in agg.columns.values]
    return agg


def plot_noise_sweep_results(summary_df: pd.DataFrame, phase: str):
    if summary_df.empty:
        print(f'No rows to plot for {phase}.')
        return
    sub = summary_df[summary_df['phase_'] == phase]
    if sub.empty:
        print(f'No rows to plot for {phase}.')
        return

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    targets = [('rmse_mean', 'RMSE'), ('pde_residual_loss_mean', 'PDE residual loss'), ('total_loss_mean', 'Total loss')]

    for ax, (col, title) in zip(axes, targets):
        for fam in sorted(sub['noise_family_'].dropna().unique()):
            s = sub[sub['noise_family_'] == fam].copy()
            s = s.sort_values('value_')
            ax.plot(s['value_'], s[col], marker='o', label=fam)
        ax.set_title(f'{phase}: {title}')
        ax.set_xlabel('Noise strength')
        ax.set_ylabel(title)
        ax.grid(alpha=0.3)
    axes[0].legend(loc='best', fontsize=8)
    plt.tight_layout()
    plt.show()


## Phase 1 and Phase 2 Baseline Under Exact vs Finite Shots

In [ ]:

def run_shot_sweep(model, phase, cfg):
    rows = []

    exact = evaluate_quantum_model_under_noise(model, phase, cfg, shots=None, noise_kwargs={}, seed=0)
    if exact['supported']:
        rows.append({
            'phase': phase, 'noise_family': 'shots', 'setting': 'exact', 'value': 0,
            'seed': 0, 'supported': True, **exact['metrics']
        })
    else:
        rows.append({'phase': phase, 'noise_family': 'shots', 'setting': 'exact', 'value': 0, 'seed': 0, 'supported': False, 'reason': exact['support'].get('reason')})

    for s in CONFIG['shots_values']:
        for seed in CONFIG['shot_seeds']:
            out = evaluate_quantum_model_under_noise(model, phase, cfg, shots=s, noise_kwargs={}, seed=seed)
            if out['supported']:
                rows.append({'phase': phase, 'noise_family': 'shots', 'setting': 'shots', 'value': s, 'seed': seed, 'supported': True, **out['metrics']})
            else:
                rows.append({'phase': phase, 'noise_family': 'shots', 'setting': 'shots', 'value': s, 'seed': seed, 'supported': False, 'reason': out['support'].get('reason')})
    return rows

rows_shots = []
rows_shots += run_shot_sweep(model_phase1, 'phase1', cfg1)
rows_shots += run_shot_sweep(model_phase2, 'phase2', cfg2)

shots_df = pd.DataFrame(rows_shots)
shots_summary = summarize_metrics([r for r in rows_shots if r.get('supported', False)])
print('Shot-sweep rows:', len(shots_df))
shots_summary.head(20)


## Phase 1 Noise Experiments (Transmittance, Detector, Phase, Source)

In [ ]:

def run_nonshot_sweeps(model, phase, cfg):
    rows = []

    for T in CONFIG['transmittance_values']:
        out = evaluate_quantum_model_under_noise(model, phase, cfg, shots=None, noise_kwargs={'transmittance': T}, seed=0)
        if out['supported']:
            rows.append({
                'phase': phase, 'noise_family': 'transmittance', 'setting': 'T', 'value': T,
                'supported': True,
                'output_dim_changed': out['distribution_change']['changed_output_dim'],
                'base_dim': out['distribution_change']['base_dim'],
                'noisy_dim': out['distribution_change']['noisy_dim'],
                'distribution_changed': out['distribution_change']['distribution_changed'],
                **out['metrics'],
            })
        else:
            rows.append({'phase': phase, 'noise_family': 'transmittance', 'setting': 'T', 'value': T, 'supported': False, 'reason': out['support'].get('reason')})

    for det in ['pnr', 'threshold']:
        out = evaluate_quantum_model_under_noise(model, phase, cfg, shots=None, noise_kwargs={}, detector_type=det, force_fock_for_detector=False, seed=0)
        if out['supported']:
            rows.append({
                'phase': phase, 'noise_family': 'detector_unbunched', 'setting': det, 'value': 0.0,
                'supported': True,
                'output_dim_changed': out['distribution_change']['changed_output_dim'],
                'base_dim': out['distribution_change']['base_dim'],
                'noisy_dim': out['distribution_change']['noisy_dim'],
                'distribution_changed': out['distribution_change']['distribution_changed'],
                **out['metrics'],
            })
        else:
            rows.append({'phase': phase, 'noise_family': 'detector_unbunched', 'setting': det, 'value': 0.0, 'supported': False, 'reason': out['support'].get('reason')})

    for det in ['pnr', 'threshold']:
        out = evaluate_quantum_model_under_noise(model, phase, cfg, shots=None, noise_kwargs={}, detector_type=det, force_fock_for_detector=True, seed=0)
        if out['supported']:
            rows.append({
                'phase': phase, 'noise_family': 'detector_fock_adapter', 'setting': det, 'value': 0.0,
                'supported': True,
                'output_dim_changed': out['distribution_change']['changed_output_dim'],
                'base_dim': out['distribution_change']['base_dim'],
                'noisy_dim': out['distribution_change']['noisy_dim'],
                'distribution_changed': out['distribution_change']['distribution_changed'],
                **out['metrics'],
            })
        else:
            rows.append({'phase': phase, 'noise_family': 'detector_fock_adapter', 'setting': det, 'value': 0.0, 'supported': False, 'reason': out['support'].get('reason')})

    for pe in CONFIG['phase_values']:
        out = evaluate_quantum_model_under_noise(model, phase, cfg, shots=None, noise_kwargs={'phase_error': pe}, seed=0)
        if out['supported']:
            rows.append({
                'phase': phase, 'noise_family': 'phase_error', 'setting': 'phase_error_rad', 'value': pe,
                'supported': True,
                'distribution_changed': out['distribution_change']['distribution_changed'],
                **out['metrics'],
            })
        else:
            rows.append({'phase': phase, 'noise_family': 'phase_error', 'setting': 'phase_error_rad', 'value': pe, 'supported': False, 'reason': out['support'].get('reason')})

    for pi in CONFIG['phase_values']:
        out = evaluate_quantum_model_under_noise(model, phase, cfg, shots=None, noise_kwargs={'phase_imprecision': pi}, seed=0)
        if out['supported']:
            rows.append({
                'phase': phase, 'noise_family': 'phase_imprecision', 'setting': 'phase_imprecision_rad', 'value': pi,
                'supported': True,
                'distribution_changed': out['distribution_change']['distribution_changed'],
                **out['metrics'],
            })
        else:
            rows.append({'phase': phase, 'noise_family': 'phase_imprecision', 'setting': 'phase_imprecision_rad', 'value': pi, 'supported': False, 'reason': out['support'].get('reason')})

    for b in CONFIG['brightness_values']:
        out = evaluate_quantum_model_under_noise(model, phase, cfg, shots=None, noise_kwargs={'brightness': b}, seed=0)
        if out['supported']:
            rows.append({'phase': phase, 'noise_family': 'brightness', 'setting': 'brightness', 'value': b, 'supported': True, 'distribution_changed': out['distribution_change']['distribution_changed'], **out['metrics']})
        else:
            rows.append({'phase': phase, 'noise_family': 'brightness', 'setting': 'brightness', 'value': b, 'supported': False, 'reason': out['support'].get('reason')})

    for ind in CONFIG['indist_values']:
        out = evaluate_quantum_model_under_noise(model, phase, cfg, shots=None, noise_kwargs={'indistinguishability': ind}, seed=0)
        if out['supported']:
            rows.append({'phase': phase, 'noise_family': 'indistinguishability', 'setting': 'indistinguishability', 'value': ind, 'supported': True, 'distribution_changed': out['distribution_change']['distribution_changed'], **out['metrics']})
        else:
            rows.append({'phase': phase, 'noise_family': 'indistinguishability', 'setting': 'indistinguishability', 'value': ind, 'supported': False, 'reason': out['support'].get('reason')})

    for g2 in CONFIG['g2_values']:
        out = evaluate_quantum_model_under_noise(model, phase, cfg, shots=None, noise_kwargs={'g2': g2}, seed=0)
        if out['supported']:
            rows.append({'phase': phase, 'noise_family': 'g2', 'setting': 'g2', 'value': g2, 'supported': True, 'distribution_changed': out['distribution_change']['distribution_changed'], **out['metrics']})
        else:
            rows.append({'phase': phase, 'noise_family': 'g2', 'setting': 'g2', 'value': g2, 'supported': False, 'reason': out['support'].get('reason')})

    return rows

rows_noise_phase1 = run_nonshot_sweeps(model_phase1, 'phase1', cfg1)
phase1_noise_df = pd.DataFrame(rows_noise_phase1)
phase1_noise_df.head(20)


## Phase 2 Noise Experiments (Same Sweeps)

In [ ]:

rows_noise_phase2 = run_nonshot_sweeps(model_phase2, 'phase2', cfg2)
phase2_noise_df = pd.DataFrame(rows_noise_phase2)
phase2_noise_df.head(20)


## Compact Results: Combined Summary, Normalization/Output-Dim Table, and Plots

In [ ]:

all_rows = []
all_rows.extend(rows_shots)
all_rows.extend(rows_noise_phase1)
all_rows.extend(rows_noise_phase2)

df_all = pd.DataFrame(all_rows)
df_all.to_csv('results/noise_study_phase1_phase2_raw.csv', index=False)

supported = df_all[df_all['supported'] == True].copy()
summary = summarize_metrics(supported.to_dict('records'))
summary.to_csv('results/noise_study_phase1_phase2_summary.csv', index=False)

print('Saved raw:', 'results/noise_study_phase1_phase2_raw.csv', 'rows=', len(df_all))
print('Saved summary:', 'results/noise_study_phase1_phase2_summary.csv', 'rows=', len(summary))

norm_cols = ['phase', 'noise_family', 'setting', 'value', 'supported', 'output_dim_changed', 'base_dim', 'noisy_dim', 'distribution_changed']
norm_table = df_all[[c for c in norm_cols if c in df_all.columns]].copy()
norm_table = norm_table.drop_duplicates()
norm_table.head(30)


In [ ]:

if not summary.empty:
    plot_noise_sweep_results(summary, 'phase1')
    plot_noise_sweep_results(summary, 'phase2')


In [ ]:

def predict_grid_phase(model, phase, nx=81, nt=81):
    model.eval()
    with torch.no_grad():
        if phase == 'phase1':
            x = torch.linspace(0, 1, nx, device=DEVICE, dtype=DTYPE).view(-1, 1)
            t = torch.linspace(0, 1, nt, device=DEVICE, dtype=DTYPE).view(-1, 1)
            X, T = torch.meshgrid(x.squeeze(), t.squeeze(), indexing='ij')
            xt = torch.stack([X.reshape(-1), T.reshape(-1)], dim=1)
            y, _ = model(xt)
            yt = exact_u1(X, T)
            return X.cpu().numpy(), T.cpu().numpy(), y.reshape(nx, nt).cpu().numpy(), yt.cpu().numpy()
        else:
            x = torch.linspace(XMIN2, XMAX2, nx, device=DEVICE, dtype=DTYPE).view(-1, 1)
            t = torch.linspace(TMIN2, TMAX2, nt, device=DEVICE, dtype=DTYPE).view(-1, 1)
            X, T = torch.meshgrid(x.squeeze(), t.squeeze(), indexing='ij')
            xt = torch.stack([X.reshape(-1), T.reshape(-1)], dim=1)
            y, _ = model(xt)
            pts = np.stack([X.detach().cpu().numpy().ravel(), T.detach().cpu().numpy().ravel()], axis=1)
            yt = interp_ref2(pts).reshape(nx, nt)
            return X.cpu().numpy(), T.cpu().numpy(), y.reshape(nx, nt).cpu().numpy(), yt

noisy_p1 = evaluate_quantum_model_under_noise(model_phase1, 'phase1', cfg1, shots=500, noise_kwargs={'transmittance': 0.85}, seed=0)
noisy_p2 = evaluate_quantum_model_under_noise(model_phase2, 'phase2', cfg2, shots=500, noise_kwargs={'transmittance': 0.85}, seed=0)

m1 = copy.deepcopy(model_phase1)
if noisy_p1['supported']:
    wrap1, _ = _make_noisy_quantum_from_base(m1.quantum, m1.feature_size, m1.quantum_output_size, shots=500, noise_kwargs={'transmittance': 0.85})
    m1.quantum = wrap1

m2 = copy.deepcopy(model_phase2)
if noisy_p2['supported']:
    wrap2, _ = _make_noisy_quantum_from_base(m2.quantum, m2.feature_size, m2.quantum_output_size, shots=500, noise_kwargs={'transmittance': 0.85})
    m2.quantum = wrap2

X1, T1, pred1_ideal, true1 = predict_grid_phase(model_phase1, 'phase1')
_, _, pred1_noisy, _ = predict_grid_phase(m1, 'phase1')
X2, T2, pred2_ideal, true2 = predict_grid_phase(model_phase2, 'phase2')
_, _, pred2_noisy, _ = predict_grid_phase(m2, 'phase2')

fig, axes = plt.subplots(2, 3, figsize=(13, 7))

axes[0,0].imshow(pred1_ideal, origin='lower', aspect='auto'); axes[0,0].set_title('Phase1 Ideal')
axes[0,1].imshow(pred1_noisy, origin='lower', aspect='auto'); axes[0,1].set_title('Phase1 Noisy (T=0.85, shots=500)')
axes[0,2].imshow(np.abs(pred1_noisy - pred1_ideal), origin='lower', aspect='auto'); axes[0,2].set_title('Phase1 |Delta|')

axes[1,0].imshow(pred2_ideal, origin='lower', aspect='auto'); axes[1,0].set_title('Phase2 Ideal')
axes[1,1].imshow(pred2_noisy, origin='lower', aspect='auto'); axes[1,1].set_title('Phase2 Noisy (T=0.85, shots=500)')
axes[1,2].imshow(np.abs(pred2_noisy - pred2_ideal), origin='lower', aspect='auto'); axes[1,2].set_title('Phase2 |Delta|')

for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


In [ ]:

def make_ranking(summary_df):
    out = []
    for phase in ['phase1', 'phase2']:
        sdf = summary_df[summary_df['phase_'] == phase].copy()
        if sdf.empty:
            continue
        base = sdf[(sdf['noise_family_'] == 'shots') & (sdf['setting_'] == 'exact')]
        if base.empty:
            continue
        base_total = float(base['total_loss_mean'].iloc[0])
        g = sdf.groupby('noise_family_')['total_loss_mean'].max().reset_index()
        g['delta_from_exact_total_loss'] = g['total_loss_mean'] - base_total
        g['phase'] = phase
        out.append(g[['phase', 'noise_family_', 'delta_from_exact_total_loss']])
    if len(out) == 0:
        return pd.DataFrame()
    rank = pd.concat(out, ignore_index=True)
    rank = rank.sort_values(['phase', 'delta_from_exact_total_loss'], ascending=[True, False])
    return rank

ranking_df = make_ranking(summary)
ranking_df



## Interpretation

Use this checklist when reading results:

1. Most damaging noise source:
   - Check `ranking_df` by phase.
2. Robustness vs fragility:
   - Flat curves in RMSE/PDE/total vs noise strength indicate robustness.
   - Steep degradation indicates fragility.
3. Phase 1 vs Phase 2 robustness:
   - Compare same noise family across phases in summary tables and plots.
4. Failure origin:
   - `shots` trend isolates sampling noise.
   - `transmittance` trend captures DV photon-loss leakage from intended photon-number sector.
   - phase/source sweeps indicate interferometer/source sensitivity.
5. Detector caveat:
   - If `detector_unbunched` shows no meaningful distribution change, detector differences are not propagating through the default UNBUNCHED path.
   - `detector_fock_adapter` provides an explicit detector-effect check in a FOCK-space adapter.


## Final Conclusion Cell

In [ ]:

supported_noise = sorted(df_all[df_all['supported'] == True]['noise_family'].dropna().unique().tolist())
unsupported_rows = df_all[df_all['supported'] == False]
unsupported_noise = sorted(unsupported_rows['noise_family'].dropna().unique().tolist())

main_finding = 'See ranking_df (largest delta_from_exact_total_loss) and summary plots.'

print('Physically correct DV noise sources tested:')
print(' - finite-shot sampling noise')
print(' - photon loss via transmittance (NoiseModel)')
print(' - detector model (PNR/threshold, with explicit support checks)')
print(' - phase noise (phase_error / phase_imprecision)')
print(' - source imperfections (brightness / indistinguishability / g2)')
print()
print('Supported by current MerLin/Perceval path:')
print(' ', supported_noise)
print()
print('Unsupported or not verified in current path (if any):')
print(' ', unsupported_noise if len(unsupported_noise) > 0 else 'None detected in this run')
print()
print('Main numerical finding:')
print(' ', main_finding)
print()
print('Recommendation for final hackathon presentation:')
print(' - Emphasize finite-shot and transmittance robustness curves first (directly physical and interpretable).')
print(' - Report detector conclusions with explicit UNBUNCHED-vs-FOCK caveat if detector effect is path-dependent.')
print(' - Avoid CV-noise claims; keep interpretation in DV photon-number language.')
